# 03 — Event Detection

The core signal processing step: detecting when appliances switch ON or OFF.

**The problem:**
We have aggregate power (one number changing over time).
We need to find the moments where a significant step-change occurs — these are appliance switching events.

**Approach: Edge detection on the power signal**
1. Smooth the signal to remove noise (median filter)
2. Compute first-order difference (rate of change)
3. Flag points where |Δpower| exceeds a threshold
4. Suppress duplicate detections within a time window

This is exactly the same principle as edge detection in image processing — we are looking for sharp transitions.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import medfilt
import warnings
warnings.filterwarnings('ignore')

from utils.signal_utils import detect_events

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
df = pd.read_csv('../data/processed/house1_7days.csv', index_col=0, parse_dates=True)
mains = df['mains']
print(f'Loaded: {len(mains)} samples, {len(mains)/3600:.1f} hours')

## Step 1: Effect of smoothing

Raw power signal has noise — sensor noise, small fluctuations.
A median filter removes these without blurring the edges we care about.

Comparing raw vs smoothed:

In [ ]:
# take 1 hour slice for visualization
slice_1h = mains.iloc[:3600]
smoothed = pd.Series(medfilt(slice_1h.values, kernel_size=5), index=slice_1h.index)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(slice_1h.index, slice_1h.values, color='steelblue', linewidth=0.6, alpha=0.8, label='Raw')
axes[0].set_title('Raw Mains Signal', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Watts')
axes[0].legend()

axes[1].plot(smoothed.index, smoothed.values, color='darkorange', linewidth=0.8, label='Median Filtered')
axes[1].set_title('After Median Filter (kernel=5)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Watts')
axes[1].set_xlabel('Time')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/03_smoothing_comparison.png', dpi=150)
plt.show()

## Step 2: Difference signal

Taking the first-order difference shows us the rate of change.
Spikes in the difference signal = step changes in power = appliance events.

In [ ]:
diff = smoothed.diff().fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
axes[0].plot(smoothed.index, smoothed.values, color='darkorange', linewidth=0.7)
axes[0].set_ylabel('Power (W)')
axes[0].set_title('Smoothed Signal', fontsize=11, fontweight='bold')

axes[1].plot(diff.index, diff.values, color='steelblue', linewidth=0.6, alpha=0.8)
axes[1].axhline(50, color='crimson', linestyle='--', linewidth=1, label='+50W threshold')
axes[1].axhline(-50, color='green', linestyle='--', linewidth=1, label='-50W threshold')
axes[1].set_ylabel('ΔPower (W/s)')
axes[1].set_title('First-Order Difference (Rate of Change)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/03_difference_signal.png', dpi=150)
plt.show()

## Step 3: Run event detection

Now apply the `detect_events()` function from our utils module.

The threshold of 50W means: ignore fluctuations below 50W, only flag real appliance events.
This is a tunable parameter — too low and you get noise events, too high and you miss small loads.

In [ ]:
# detect events on full 7-day mains signal
events = detect_events(mains, threshold=50, min_gap_seconds=3)

print(f'Events detected: {len(events)}')
print(f'ON events: {(events["direction"]=="ON").sum()}')
print(f'OFF events: {(events["direction"]=="OFF").sum()}')
events.head(10)

## Visualize detected events on the signal

In [ ]:
# plot 6 hours with events marked
slice_6h = mains.iloc[:6*3600]
events_6h = events[events['timestamp'] <= slice_6h.index[-1]]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(slice_6h.index, slice_6h.values, color='steelblue', linewidth=0.7, alpha=0.8, zorder=1)

# mark ON events green, OFF events red
on_events  = events_6h[events_6h['direction'] == 'ON']
off_events = events_6h[events_6h['direction'] == 'OFF']

for ts in on_events['timestamp']:
    ax.axvline(ts, color='green', alpha=0.5, linewidth=0.8)
for ts in off_events['timestamp']:
    ax.axvline(ts, color='crimson', alpha=0.5, linewidth=0.8)

patch_on  = mpatches.Patch(color='green', alpha=0.5, label=f'ON events ({len(on_events)})')
patch_off = mpatches.Patch(color='crimson', alpha=0.5, label=f'OFF events ({len(off_events)})')
ax.legend(handles=[patch_on, patch_off], fontsize=9)
ax.set_ylabel('Power (Watts)', fontsize=11)
ax.set_title('Detected Events on Mains Signal (6 hours)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_detected_events.png', dpi=150)
plt.show()

## Effect of threshold on event count

How sensitive is our detector to the threshold choice? Let's check:

In [ ]:
thresholds = [20, 30, 50, 75, 100, 150, 200]
counts = []
for t in thresholds:
    ev = detect_events(mains, threshold=t)
    counts.append(len(ev))
    print(f'Threshold {t:4d}W -> {len(ev):4d} events')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, counts, marker='o', color='steelblue', linewidth=2)
ax.axvline(50, color='crimson', linestyle='--', linewidth=1.5, label='Chosen threshold (50W)')
ax.set_xlabel('Threshold (Watts)', fontsize=11)
ax.set_ylabel('Events Detected', fontsize=11)
ax.set_title('Event Count vs Detection Threshold', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/03_threshold_sensitivity.png', dpi=150)
plt.show()

In [ ]:
# save events for next notebook
events.to_csv('../data/processed/detected_events.csv', index=False)
print(f'Saved {len(events)} events.')